# AID2026 — Incident Detection Training Notebook
Run each cell **in order**, top to bottom.

**First:** Set runtime to A100 → Runtime → Change runtime type → A100

## Cell 1 — Download Dataset from Shared Drive
Downloads all videos and CSVs directly from the competition shared folder into Colab. Takes 5–15 minutes.

In [32]:
import os, zipfile, time
!pip install -q gdown
import gdown

# Create folder structure
os.makedirs('/content/aid2026/data/train', exist_ok=True)
os.makedirs('/content/aid2026/data/val',   exist_ok=True)

# --- CSVs (download in seconds) ---
print('Downloading train_GT.csv...')
gdown.download(id='1gktt4ZlS75ijx50ONbiFRkwWZkyb9b2l', output='/content/aid2026/data/train_GT.csv', quiet=False)

print('Downloading val_GT.csv...')
gdown.download(id='1h1f4aMyMH845t5yrcllglXFSs5E_lX_2', output='/content/aid2026/data/val_GT.csv', quiet=False)

# --- Train videos ---
print('\nDownloading train.zip...')
t0 = time.time()
gdown.download(id='1-ffvPj8aGUUUnb4tzpp8nrPK4Kl-nHUF', output='/content/aid2026/data/train.zip', quiet=False)
print(f'Downloaded in {(time.time()-t0)/60:.1f} mins. Unzipping...')
with zipfile.ZipFile('/content/aid2026/data/train.zip', 'r') as z:
    z.extractall('/content/aid2026/data/train/')
os.remove('/content/aid2026/data/train.zip')
print(f'Train videos: {len(os.listdir("/content/aid2026/data/train"))} files')

# --- Val videos ---
print('\nDownloading val.zip...')
t0 = time.time()
gdown.download(id='1KOAbJg1yL5AnK9nMN8qesNJeCQX-O0Gc', output='/content/aid2026/data/val.zip', quiet=False)
print(f'Downloaded in {(time.time()-t0)/60:.1f} mins. Unzipping...')
with zipfile.ZipFile('/content/aid2026/data/val.zip', 'r') as z:
    z.extractall('/content/aid2026/data/val/')
os.remove('/content/aid2026/data/val.zip')
print(f'Val videos: {len(os.listdir("/content/aid2026/data/val"))} files')

print('\nDataset ready!')

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
The folder you are executing pip from can no longer be found.


Downloading...
From: https://drive.google.com/uc?id=1gktt4ZlS75ijx50ONbiFRkwWZkyb9b2l
To: /content/aid2026/data/train_GT.csv
100%|██████████| 31.7k/31.7k [00:00<00:00, 75.2MB/s]


Downloading...
From: https://drive.google.com/uc?id=1h1f4aMyMH845t5yrcllglXFSs5E_lX_2
To: /content/aid2026/data/val_GT.csv
100%|██████████| 7.06k/7.06k [00:00<00:00, 24.2MB/s]


Downloading...
From (original): https://drive.google.com/uc?id=1-ffvPj8aGUUUnb4tzpp8nrPK4Kl-nHUF
From (redirected): https://drive.google.com/uc?id=1-ffvPj8aGUUUnb4tzpp8nrPK4Kl-nHUF&confirm=t&uuid=00334dd8-be38-4530-a1ac-320bd56d518f
To: /content/aid2026/data/train.zip
100%|██████████| 4.45G/4.45G [02:19<00:00, 31.8MB/s]


Downloaded in 2.4 mins. Unzipping...
Train videos: 2 files



Downloading...
From (original): https://drive.google.com/uc?id=1KOAbJg1yL5AnK9nMN8qesNJeCQX-O0Gc
From (redirected): https://drive.google.com/uc?id=1KOAbJg1yL5AnK9nMN8qesNJeCQX-O0Gc&confirm=t&uuid=476ff046-0018-4d62-af53-58a8c1026448
To: /content/aid2026/data/val.zip
100%|██████████| 1.62G/1.62G [00:29<00:00, 54.8MB/s]


Downloaded in 0.5 mins. Unzipping...
Val videos: 2 files

Dataset ready!


## Cell 2 — Upload Project Code


In [43]:
print('Downloading aid2026_code.zip from Drive...')
gdown.download(id='11-qWltCJCxPKO5IGAuZZ-GUeuWwcHxgJ', output='/content/aid2026_code.zip', quiet=False)

print('Unzipping...')
with zipfile.ZipFile('/content/aid2026_code.zip', 'r') as z:
    z.extractall('/content/aid2026/')
os.remove('/content/aid2026_code.zip')

print('Code ready. Files:')
for item in sorted(os.listdir('/content/aid2026')):
    print(f'  {item}')

#https://drive.google.com/file/d/11-qWltCJCxPKO5IGAuZZ-GUeuWwcHxgJ/view?usp=drive_link

Downloading...
From: https://drive.google.com/uc?id=11-qWltCJCxPKO5IGAuZZ-GUeuWwcHxgJ
To: /content/aid2026_code.zip
100%|██████████| 13.6k/13.6k [00:00<00:00, 37.4MB/s]

Unzipping...
Code ready. Files:
  adi2026_code
  data


## Cell 3 — Check GPU

In [1]:
import torch
print(f'PyTorch version : {torch.__version__}')
print(f'GPU available   : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU name        : {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM            : {vram:.1f} GB')
else:
    print('WARNING: No GPU — go to Runtime → Change runtime type → A100')

PyTorch version : 2.10.0+cu128
GPU available   : True
GPU name        : Tesla T4
VRAM            : 15.6 GB


## Cell 4 — Install Dependencies
Takes about 1 minute.

In [2]:
!pip install -q transformers >=4.41.0 accelerate decord scikit-learn einops timm opencv-python-headless
print('All dependencies installed.')

All dependencies installed.


## Cell 5 — Verify Everything is in Place

In [3]:
import os, sys

PROJECT_DIR = '/content/aid2026'
DATA_DIR    = os.path.join(PROJECT_DIR, 'data')
sys.path.insert(0, os.path.join(PROJECT_DIR, 'src'))
os.chdir(PROJECT_DIR)

expected = [
    'data/train_GT.csv',
    'data/val_GT.csv',
    'data/train',
    'data/val',
    'src/model.py',
    'src/dataset.py',
    'src/train.py',
    'test.py',
]

all_ok = True
for path in expected:
    exists = os.path.exists(os.path.join(PROJECT_DIR, path))
    icon = 'OK     ' if exists else 'MISSING'
    print(f'  [{icon}]  {path}')
    if not exists:
        all_ok = False

print()
print('Ready to train!' if all_ok else 'Fix missing files before continuing.')

  [OK     ]  data/train_GT.csv
  [OK     ]  data/val_GT.csv
  [OK     ]  data/train
  [OK     ]  data/val
  [OK     ]  src/model.py
  [OK     ]  src/dataset.py
  [OK     ]  src/train.py
  [OK     ]  test.py

Ready to train!


In [4]:
import os, glob

# Find actual video files
for root, dirs, files in os.walk(f'{DATA_DIR}/train'):
    for f in files[:3]:  # show first 3 files found
        print(os.path.join(root, f))
    if files:
        break

/content/aid2026/data/train/train/video132.mp4
/content/aid2026/data/train/train/t917.mp4
/content/aid2026/data/train/train/s56.mp4


## Cell 6 — Sanity Check: Load One Video
Confirms video filenames in the CSV match the actual files on disk.

In [5]:
import pandas as pd
from dataset import read_video_frames

# Read annotation files (semicolon-separated)
train_df = pd.read_csv(f'{DATA_DIR}/train_GT.csv', sep=';')
val_df   = pd.read_csv(f'{DATA_DIR}/val_GT.csv', sep=';')

print(f'Train rows: {len(train_df)} | Columns: {list(train_df.columns)}')
print('\nSample rows:')
print(train_df.head(3).to_string())

# First video ID from CSV
first_id = str(train_df.iloc[0]['Id Video']).strip()
print(f'\nTrying to load video ID: "{first_id}"')

# Search both possible folder structures
candidates = (
    glob.glob(f'{DATA_DIR}/train/{first_id}') +
    glob.glob(f'{DATA_DIR}/train/{first_id}.*') +
    glob.glob(f'{DATA_DIR}/train/train/{first_id}') +
    glob.glob(f'{DATA_DIR}/train/train/{first_id}.*')
)

if candidates:
    video_path = candidates[0]
    print(f'Found video: {video_path}')

    frames, fps = read_video_frames(video_path, max_frames=32)

    print(f'SUCCESS: {len(frames)} frames at {fps:.1f} fps')
    print(f'Frame shape: {frames[0].shape}')
else:
    print('\nERROR: Video not found.')

    sample_files = (
        glob.glob(f'{DATA_DIR}/train/*')[:5] +
        glob.glob(f'{DATA_DIR}/train/train/*')[:5]
    )

    print('\nSample filenames:')
    print([os.path.basename(f) for f in sample_files])

    print('\nCSV Id Video values must match these filenames exactly.')

Train rows: 1246 | Columns: ['Id Video', 'Duration', 'Start', 'End']

Sample rows:
  Id Video Duration  Start    End
0       v1    00:23  00:15  00:20
1       v2    00:04  00:03  00:04
2       v3    00:14  00:05  00:09

Trying to load video ID: "v1"
Found video: /content/aid2026/data/train/train/v1.mp4
SUCCESS: 32 frames at 59.5 fps
Frame shape: (1358, 2386, 3)


## Cell 7 — Configure Training
Safe to leave all settings as-is for the first run.

In [6]:
import os
import json

cfg = {
    # Backbone
    'backbone': 'MCG-NJU/videomae-small-finetuned-kinetics',

    # Video sampling
    'clip_frames': 16,
    'clip_hop': 8,
    'max_clips': 24,          # reduced from 48
    'context_clips': 4,       # reduced from 8

    # Spatial preprocessing
    'image_size': 224,

    # Model regularization
    'dropout': 0.3,
    'freeze_layers': 10,      # freeze more initially

    # Data
    'train_csv': f'{DATA_DIR}/train_GT.csv',
    'val_csv': f'{DATA_DIR}/val_GT.csv',

    # IMPORTANT: actual video folders
    'train_videos_dir': f'{DATA_DIR}/train/train',
    'val_videos_dir': f'{DATA_DIR}/val/val',

    # Training
    'batch_size': 2,          # safer for Colab
    'num_workers': 2,
    'epochs': 25,
    'lr': 1e-4,               # slightly safer
    'weight_decay': 1e-4,
    'patience': 6,
    'amp': True,

    # Output
    'checkpoint_dir': f'{PROJECT_DIR}/checkpoints',
}

os.makedirs(f'{PROJECT_DIR}/configs', exist_ok=True)
os.makedirs(cfg['checkpoint_dir'], exist_ok=True)

cfg_path = f'{PROJECT_DIR}/configs/train_config.json'

with open(cfg_path, 'w') as f:
    json.dump(cfg, f, indent=2)

print("Config saved.")
print(f'Backbone: {cfg["backbone"]}')
print(f'Batch size: {cfg["batch_size"]}')
print(f'Max clips: {cfg["max_clips"]}')
print(f'Context clips: {cfg["context_clips"]}')
print(f'Train dir: {cfg["train_videos_dir"]}')
print(f'Val dir: {cfg["val_videos_dir"]}')

Config saved.
Backbone: MCG-NJU/videomae-small-finetuned-kinetics
Batch size: 2
Max clips: 24
Context clips: 4
Train dir: /content/aid2026/data/train/train
Val dir: /content/aid2026/data/val/val


In [14]:
with open("/content/aid2026/src/train.py", "r") as f:
    print(f.read())

"""
AID2026 — Training Loop
Features:
  - Mixed precision (bfloat16) for Colab A100/T4
  - Cosine LR schedule with warm-up
  - Threshold optimization on validation set
  - Early stopping on F1
  - Gradient clipping
  - Checkpoint saving (best F1)
"""

import os
import time
import json
import math
import argparse
from pathlib import Path
from typing import Dict, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from torch.cuda.amp import GradScaler, autocast
from sklearn.metrics import f1_score, precision_score, recall_score

from model import IncidentDetector, OnsetDetector
from dataset import build_dataloaders


# ---------------------------------------------------------------------------
# Metrics
# ---------------------------------------------------------------------------

def compute_metrics(
    all_scores: np.ndarray,    # (N, max_clips)
    all_labels: np.ndarray,    # (N,)
    al

In [81]:
%%writefile /content/aid2026/src/dataset.py
"""
AID2026 — Dataset & DataLoader
Handles MIVIA-AID video loading, clip extraction, and augmentation.
"""

import os
import csv
import math
import random
from pathlib import Path
from typing import List, Dict, Tuple

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.v2 as T

# Try decord first (fast), fall back to torchvision
try:
    from decord import VideoReader, cpu
    DECORD_AVAILABLE = True
except ImportError:
    DECORD_AVAILABLE = False
    import torchvision.io as tvio


# ---------------------------------------------------------------------------
# CONSTANTS
# ---------------------------------------------------------------------------

CLIP_FRAMES  = 16
CLIP_STRIDE  = 8
SPATIAL_SIZE = 224
CLIP_HOP     = 8
MAX_CLIPS    = 64


# ---------------------------------------------------------------------------
# VIDEO LOADING
# ---------------------------------------------------------------------------

def read_video_frames(path: str, max_frames: int = 4096):
    if DECORD_AVAILABLE:
        vr = VideoReader(path, ctx=cpu(0))
        fps = vr.get_avg_fps()
        n = min(len(vr), max_frames)
        frames = vr.get_batch(list(range(n))).asnumpy()
        return frames, fps
    else:
        video, _, info = tvio.read_video(path, pts_unit="sec")
        fps = info.get("video_fps", 25.0)
        frames = video.numpy()
        if len(frames) > max_frames:
            frames = frames[:max_frames]
        return frames, fps


def extract_clips(frames, clip_frames=CLIP_FRAMES, clip_stride=CLIP_STRIDE,
                  clip_hop=CLIP_HOP, max_clips=MAX_CLIPS):

    T = len(frames)
    starts = list(range(0, T - clip_frames * clip_stride + 1, clip_hop))

    if len(starts) == 0:
        starts = [0]

    starts = starts[:max_clips]

    clips = []
    for s in starts:
        indices = [min(s + i * clip_stride, T - 1) for i in range(clip_frames)]
        clips.append(frames[indices])

    return np.stack(clips), starts


# ---------------------------------------------------------------------------
# TRANSFORMS
# ---------------------------------------------------------------------------

def build_transforms(train: bool, size=SPATIAL_SIZE):
    ops = []

    if train:
        ops += [
            T.RandomResizedCrop(size, scale=(0.7, 1.0), antialias=True),
            T.RandomHorizontalFlip(),
            T.ColorJitter(0.3, 0.3, 0.2, 0.05),
        ]
    else:
        ops += [
            T.Resize(int(size * 1.1), antialias=True),
            T.CenterCrop(size),
        ]

    ops += [
        T.ToDtype(torch.float32, scale=True),
        T.Normalize([0.485, 0.456, 0.406],
                    [0.229, 0.224, 0.225]),
    ]

    return T.Compose(ops)


# ---------------------------------------------------------------------------
# DATASET
# ---------------------------------------------------------------------------

class MIVIADataset(Dataset):

    def __init__(
        self,
        csv_path: str,
        videos_dir: str,
        train: bool = True,
        clip_frames: int = CLIP_FRAMES,
        clip_stride: int = CLIP_STRIDE,
        clip_hop: int = CLIP_HOP,
        max_clips: int = MAX_CLIPS,
    ):
        self.videos_dir = Path(videos_dir)
        self.train = train

        self.clip_frames = clip_frames
        self.clip_stride = clip_stride
        self.clip_hop = clip_hop
        self.max_clips = max_clips

        self.transform = build_transforms(train)

        self.samples = self._load_csv(csv_path)

        print(
            f"[Dataset] Loaded {len(self.samples)} samples | "
            f"positives: {sum(s['label'] for s in self.samples)}"
        )

    # ---------------- FIXED CSV LOADER ----------------
    def _load_csv(self, path: str) -> List[Dict]:

        samples = []

        with open(path, newline="", encoding="utf-8-sig") as f:
            reader = csv.DictReader(f, delimiter=";")

            # clean headers (THIS FIXES YOUR ERROR)
            reader.fieldnames = [h.strip() for h in reader.fieldnames]

            for row in reader:
                row = {k.strip(): v for k, v in row.items()}

                # flexible column matching
                vid = (
                    row.get("Id Video")
                    or row.get("Id_Video")
                    or row.get("Video")
                    or row.get("video_id")
                )

                if vid is None:
                    raise ValueError(
                        f"CSV missing video column. Available: {list(row.keys())}"
                    )

                duration = float(row.get("Duration", 0) or 0)

                start_str = (row.get("Start") or "").strip()
                end_str = (row.get("End") or "").strip()

                label = 1 if start_str else 0
                onset_sec = float(start_str) if start_str else -1.0
                end_sec = float(end_str) if end_str else -1.0

                samples.append({
                    "video_id": vid.strip(),
                    "duration": duration,
                    "label": label,
                    "onset_sec": onset_sec,
                    "end_sec": end_sec,
                })

        return samples

    # --------------------------------------------------

    def __len__(self):
        return len(self.samples)

    def _find_video(self, video_id: str) -> Path:
        search_dirs = [
            self.videos_dir,
            self.videos_dir / "train",
            self.videos_dir / "val",
        ]

        for folder in search_dirs:
            for ext in ["", ".mp4", ".avi", ".mov", ".mkv"]:
                p = folder / f"{video_id}{ext}"
                if p.exists():
                    return p

        raise FileNotFoundError(f"Video not found: {video_id}")

    def __getitem__(self, idx: int):

        sample = self.samples[idx]
        vid_path = self._find_video(sample["video_id"])

        frames, fps = read_video_frames(str(vid_path))

        clips_np, start_frames = extract_clips(
            frames,
            self.clip_frames,
            self.clip_stride,
            self.clip_hop,
            self.max_clips,
        )

        # onset clip
        if sample["label"] == 1 and sample["onset_sec"] >= 0:
            onset_frame = int(sample["onset_sec"] * fps)
            dists = [abs(s - onset_frame) for s in start_frames]
            onset_clip = int(np.argmin(dists))
        else:
            onset_clip = -1

        clips_tensor = self._apply_transforms(clips_np)

        return {
            "clips": clips_tensor,
            "label": torch.tensor(sample["label"], dtype=torch.long),
            "onset_clip": torch.tensor(onset_clip, dtype=torch.long),
            "onset_sec": torch.tensor(sample["onset_sec"], dtype=torch.float),
            "fps": torch.tensor(fps, dtype=torch.float),
            "start_frames": torch.tensor(start_frames, dtype=torch.long),
            "video_id": sample["video_id"],
        }

    def _apply_transforms(self, clips_np):
        N, T, H, W, C = clips_np.shape
        out = []

        for n in range(N):
            frames = torch.from_numpy(clips_np[n]).permute(0, 3, 1, 2)

            t_frames = []
            for f in range(T):
                t_frames.append(self.transform(frames[f]))

            clip = torch.stack(t_frames, dim=1)  # (3, T, H, W)
            out.append(clip)

        return torch.stack(out)


# ---------------------------------------------------------------------------
# COLLATE
# ---------------------------------------------------------------------------

def collate_fn(batch):

    max_clips = max(b["clips"].shape[0] for b in batch)
    C, T, H, W = batch[0]["clips"].shape[1:]

    clips_padded = []
    masks = []

    for b in batch:
        n = b["clips"].shape[0]

        pad = max_clips - n
        if pad > 0:
            padding = torch.zeros(pad, C, T, H, W)
            clips_padded.append(torch.cat([b["clips"], padding], dim=0))
        else:
            clips_padded.append(b["clips"])

        mask = torch.zeros(max_clips, dtype=torch.bool)
        mask[:n] = True
        masks.append(mask)

    return {
        "clips": torch.stack(clips_padded),
        "padding_mask": torch.stack(masks),
        "label": torch.stack([b["label"] for b in batch]),
        "onset_clip": torch.stack([b["onset_clip"] for b in batch]),
        "onset_sec": torch.stack([b["onset_sec"] for b in batch]),
        "fps": torch.stack([b["fps"] for b in batch]),
        "start_frames": [b["start_frames"] for b in batch],
        "video_ids": [b["video_id"] for b in batch],
    }


# ---------------------------------------------------------------------------
# DATALOADERS
# ---------------------------------------------------------------------------

def build_dataloaders(train_csv, val_csv, videos_dir,
                       batch_size=4, num_workers=2, **kwargs):

    train_ds = MIVIADataset(train_csv, videos_dir, train=True, **kwargs)
    val_ds   = MIVIADataset(val_csv, videos_dir, train=False, **kwargs)

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        collate_fn=collate_fn,
        pin_memory=True,
        drop_last=True,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        collate_fn=collate_fn,
        pin_memory=True,
    )

    return train_loader, val_loader


Overwriting /content/aid2026/src/dataset.py


In [15]:
import pandas as pd

df = pd.read_csv(cfg["train_csv"], sep=";")
df.columns = df.columns.str.strip()

print(df.head())
print(df.columns)

  Id Video Duration  Start    End
0       v1    00:23  00:15  00:20
1       v2    00:04  00:03  00:04
2       v3    00:14  00:05  00:09
3       v4    00:09  00:02  00:06
4       v5    00:11  00:04  00:06
Index(['Id Video', 'Duration', 'Start', 'End'], dtype='object')


In [16]:
def time_to_seconds(x):
    if x is None or x == "":
        return 0.0

    x = str(x)

    try:
        return float(x)
    except:
        pass

    if ":" in x:
        parts = x.split(":")
        if len(parts) == 2:
            m, s = parts
            return int(m) * 60 + float(s)
        if len(parts) == 3:
            h, m, s = parts
            return int(h)*3600 + int(m)*60 + float(s)

    return 0.0

In [17]:
import numpy as np
from pathlib import Path
import torch

class NotebookDataset(torch.utils.data.Dataset):

    def __init__(self, df, videos_dir):
        self.df = df
        self.videos_dir = Path(videos_dir)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        vid = row["Id Video"]

        duration = time_to_seconds(row.get("Duration"))

        start = row.get("Start")
        end = row.get("End")

        label = 1 if not pd.isna(start) and str(start) != "" else 0

        onset = time_to_seconds(start) if label else -1

        return {
            "video_id": vid,
            "duration": duration,
            "label": torch.tensor(label),
            "onset_sec": torch.tensor(onset),
        }

In [18]:
train_ds = NotebookDataset(df, cfg["train_videos_dir"])

train_ds[0]

{'video_id': 'v1',
 'duration': 23.0,
 'label': tensor(1),
 'onset_sec': tensor(15.)}

In [19]:
from dataset import read_video_frames

In [20]:
from decord import VideoReader, cpu
import numpy as np

def safe_read_video(path, max_frames=256):
    vr = VideoReader(path, ctx=cpu(0))
    fps = vr.get_avg_fps()

    n = min(len(vr), max_frames)

    # DO NOT use full index list for long videos
    indices = np.linspace(0, n-1, n).astype(int)

    frames = vr.get_batch(indices).asnumpy()
    return frames, fps

In [21]:
sample_vid = df.iloc[0]["Id Video"]

video_path = None
for ext in ["", ".mp4", ".avi", ".mov", ".mkv"]:
    p = Path(cfg["train_videos_dir"]) / f"{sample_vid}{ext}"
    if p.exists():
        video_path = p
        break

frames, fps = safe_read_video(str(video_path))

print(frames.shape, fps)

(256, 1358, 2386, 3) 59.532908704883226


In [23]:
import pandas as pd

def fixed_load_csv(path):
    df = pd.read_csv(path, sep=";")
    df.columns = df.columns.str.strip()

    samples = []

    for _, row in df.iterrows():
        vid = row["Id Video"]
        duration = str(row["Duration"])

        # FIX: time format like "00:23"
        def to_seconds(t):
            if pd.isna(t) or t == "":
                return -1
            if ":" in str(t):
                m, s = str(t).split(":")
                return int(m) * 60 + int(s)
            return float(t)

        start = to_seconds(row.get("Start", ""))
        end = to_seconds(row.get("End", ""))

        samples.append({
            "video_id": vid,
            "duration": to_seconds(duration),
            "label": 1 if start > 0 else 0,
            "onset_sec": start,
            "end_sec": end,
        })

    return samples

In [24]:
from dataset import MIVIADataset

MIVIADataset._load_csv = staticmethod(fixed_load_csv)

In [25]:
train_ds = MIVIADataset(
    csv_path=cfg["train_csv"],
    videos_dir=cfg["train_videos_dir"],
    train=True,
    clip_frames=cfg["clip_frames"],
    clip_stride=8,
    clip_hop=cfg["clip_hop"],
    max_clips=cfg["max_clips"]
)

[Dataset] Loaded 1246 samples | positives: 662


In [26]:
from dataset import build_dataloaders
from model import IncidentDetector
from train import train
import torch

In [27]:
import inspect
print(inspect.signature(IncidentDetector))

(backbone_name: str = 'MCG-NJU/videomae-small', num_frames: int = 16, tubelet_size: int = 2, context_clips: int = 8, dropout: float = 0.3, freeze_backbone_layers: int = 8)


In [28]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_instance = IncidentDetector(
    backbone_name=cfg["backbone"],
    num_frames=cfg["clip_frames"],
    context_clips=cfg["context_clips"],
    dropout=cfg["dropout"],
    freeze_backbone_layers=cfg["freeze_layers"]
).to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/182 [00:00<?, ?it/s]

VideoMAEModel LOAD REPORT from: MCG-NJU/videomae-small-finetuned-kinetics
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
fc_norm.bias      | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 
fc_norm.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [31]:
train_loader, val_loader = build_dataloaders(
    train_csv=cfg["train_csv"],
    val_csv=cfg["val_csv"],
    videos_dir=cfg["train_videos_dir"],
    batch_size=2,
    num_workers=0,
    clip_frames=8,
    clip_stride=8,
    clip_hop=cfg["clip_hop"],
    max_clips=12,
)

[Dataset] Loaded 1246 samples | positives: 662
[Dataset] Loaded 310 samples | positives: 105


In [32]:
batch = next(iter(train_loader))

print(type(batch))
print(batch.keys() if isinstance(batch, dict) else len(batch))

<class 'dict'>
dict_keys(['clips', 'padding_mask', 'label', 'onset_clip', 'onset_sec', 'fps', 'start_frames', 'video_ids'])


In [33]:
device = next(model_instance.parameters()).device

batch = next(iter(train_loader))

clips = batch["clips"].to(device)
labels = batch["label"].float().to(device)

out = model_instance(clips)

print("output shape:", out.shape)
print("label shape:", labels.shape)

: 

: 

: 

## Cell 8 — Train the Model
**Takes 2–4 hours. Do not close the tab.**

You will see output like this every 20 steps:
```
[E1 S0/415] loss=0.6821 lr=2.00e-04 t=12.3s
[Epoch 1] F1=0.6123 P=0.6500 R=0.5800 | delay=3.2s
[Checkpoint] Saved  (F1=0.6123)
```
The best model saves automatically whenever F1 improves.

In [108]:
import json
from train import train

# load your config you already saved
cfg_path = f'{PROJECT_DIR}/configs/train_config.json'
cfg = json.load(open(cfg_path))

# start training
train(cfg)

[Trainer] Device: cuda


ValueError: CSV missing video column. Available: ['Id Video;Duration;Start;End']

In [92]:
import importlib
import dataset
importlib.reload(dataset)

<module 'dataset' from '/content/aid2026/src/dataset.py'>

In [ ]:
# %load /content/aid2026/src/dataset.py
# %%writefile /content/aid2026/src/dataset.py

## Cell 9 — Plot Training Curves
Run after training finishes to see how F1 improved over time.

In [ ]:
import json, matplotlib.pyplot as plt

epochs, f1s, precs, recs, delays = [], [], [], [], []
with open(f'{PROJECT_DIR}/checkpoints/metrics_log.jsonl') as f:
    for line in f:
        m = json.loads(line)
        epochs.append(m['epoch']); f1s.append(m['f1'])
        precs.append(m['precision']); recs.append(m['recall'])
        delays.append(m['avg_delay_sec'])

best_idx = f1s.index(max(f1s))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(epochs, f1s,   label='F1',       linewidth=2, color='green')
axes[0].plot(epochs, precs, label='Precision', linewidth=2, color='blue',   linestyle='--')
axes[0].plot(epochs, recs,  label='Recall',    linewidth=2, color='orange', linestyle=':')
axes[0].axvline(x=epochs[best_idx], color='green', alpha=0.3, label=f'Best epoch ({epochs[best_idx]})')
axes[0].set_title('Validation Metrics'); axes[0].set_ylim(0,1); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(epochs, delays, color='red', linewidth=2)
axes[1].set_title('Avg Notification Delay (s)'); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f'Best F1    : {f1s[best_idx]:.4f}  (epoch {epochs[best_idx]})')
print(f'Precision  : {precs[best_idx]:.4f}')
print(f'Recall     : {recs[best_idx]:.4f}')
print(f'Avg Delay  : {delays[best_idx]:.2f}s')

## Cell 10 — Save Checkpoint to Google Drive ⚠️
**Run before closing Colab.** Saves your trained model to Drive so you don't lose it.

In [ ]:
from google.colab import drive
import shutil
drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/AID2026/checkpoints'
os.makedirs(SAVE_DIR, exist_ok=True)

ckpt = f'{PROJECT_DIR}/checkpoints/checkpoint_best.pt'
if os.path.exists(ckpt):
    shutil.copy(ckpt, f'{SAVE_DIR}/checkpoint_best.pt')
    shutil.copy(f'{PROJECT_DIR}/checkpoints/metrics_log.jsonl', f'{SAVE_DIR}/metrics_log.jsonl')
    print(f'Saved to Drive: {SAVE_DIR}')
else:
    print('Checkpoint not found. Has Cell 8 finished?')

## Cell 11 — Package & Save Submission to Drive

In [ ]:
import shutil, glob
from google.colab import drive
drive.mount('/content/drive')

SUB = '/content/submission'
if os.path.exists(SUB): shutil.rmtree(SUB)
os.makedirs(SUB)

shutil.copy(f'{PROJECT_DIR}/test.py', f'{SUB}/test.py')
shutil.copytree(f'{PROJECT_DIR}/src', f'{SUB}/src')
shutil.copytree(f'{PROJECT_DIR}/checkpoints', f'{SUB}/checkpoints')
nb = glob.glob('/content/*.ipynb')
if nb: shutil.copy(nb[0], f'{SUB}/test.ipynb')

zip_path = '/content/drive/MyDrive/AID2026/aid2026_submission'
shutil.make_archive(zip_path, 'zip', SUB)
size_mb = os.path.getsize(zip_path + '.zip') / 1e6
print(f'Submission saved to Drive ({size_mb:.0f} MB)')
print(f'Location: {zip_path}.zip')